In [7]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"


In [10]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py

--2026-07-22 17:29:27--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py.1’

ingest.py.1         100%[===================>]     738  --.-KB/s    in 0s      

2026-07-22 17:29:28 (9.76 MB/s) - ‘ingest.py.1’ saved [738/738]

--2026-07-22 17:29:29--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 

In [13]:
from ingest import load_faq_data
documents = load_faq_data()

In [15]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

118

In [16]:
documents = documents_llm

In [17]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [19]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [20]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

- We ask the LLM to use different wording from the original document. This makes the evaluation more realistic - real users won't phrase their questions the same way as the FAQ.

In [28]:
import os 
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
llm_model = "openai/gpt-oss-20b"


In [29]:
import json

user_prompt = json.dumps(doc)

In [30]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [31]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [32]:
response = openai_client.responses.parse(
    model=llm_model,
    input=messages,
    text_format=Questions
)

In [33]:
result = response.output_parsed

print(result)

questions=['Can I still enroll after seeing the course recently?', 'If I join late, can I still earn the certificate?', 'What’s the deadline to submit the final project for certification?', 'Do late registrants need to finish the project before a specific date?', 'Is it possible to get the certificate if I submit the project after enrollment?']


In [34]:
print(result.questions)

['Can I still enroll after seeing the course recently?', 'If I join late, can I still earn the certificate?', 'What’s the deadline to submit the final project for certification?', 'Do late registrants need to finish the project before a specific date?', 'Is it possible to get the certificate if I submit the project after enrollment?']


In [37]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions,
    model=llm_model,
)

print(result.questions)

['Can I still enroll now that I’ve found out about the course?', 'If I join late, can I still earn a certificate?', 'How do I submit my project to qualify for a certificate?', 'What’s the deadline for submitting the final project for certification?', 'Will I be awarded a certificate if I finish the project after the deadline?']


In [38]:
usage.input_tokens, usage.output_tokens

(336, 731)

In [39]:
from evaluation_utils import calc_price

In [40]:
cost = calc_price(usage)

cost

{'input_cost': 0.000252, 'output_cost': 0.0032895, 'total_cost': 0.0035415}

In [41]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still enroll now that I’ve found out about the course?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, can I still earn a certificate?',
  'document': '74eb249bbf'},
 {'question': 'How do I submit my project to qualify for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for submitting the final project for certification?',
  'document': '74eb249bbf'},
 {'question': 'Will I be awarded a certificate if I finish the project after the deadline?',
  'document': '74eb249bbf'}]

## Generating Ground Truth for All Documents

In [42]:
!pipenv install tqdm pandas

Loading .env environment variables...
Courtesy Notice:
Pipenv found itself running within a virtual environment,  so it will 
automatically use that environment, instead of  creating its own for any 
project. You can set
PIPENV_IGNORE_VIRTUALENVS=1 to force pipenv to ignore that environment and 
create  its own instead.
You can set PIPENV_VERBOSITY=-1 to suppress this warning.
To activate this project's virtualenv, run pipenv shell.
Alternatively, run a command inside the virtualenv with pipenv run.
Installing tqdm...
✔ Installation Succeeded
Installing pandas...
✔ Installation Succeeded
To activate this project's virtualenv, run pipenv shell.
Alternatively, run a command inside the virtualenv with pipenv run.
Installing dependencies from Pipfile.lock (03e986)...
All dependencies are now up-to-date!
Upgrading tqdm, pandas in  dependencies.
Building requirements...
Resolving dependencies....
✔ Success! Locking packages...
⠼ Locking packages...Warning: WARNING:pipenv.patched.pip._vendor.

In [43]:
from evaluation_utils import llm_structured_retry

In [44]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model=llm_model
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [45]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

## Parallel processing

In [47]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

# This submits one job per document, updates the progress bar when a job finishes, and collects the results.

In [48]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/118 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01ks7h6m1ffk3tt9zb0yx6496w` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7220, Requested 835. Please try again in 412.5ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

In [ ]:
# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

# wget -O data/ground_truth-new.csv ${PREFIX}/04-evaluation/data/ground_truth-new.csv
!cp "/home/hsu/Documents/Learning/llm-zoomcamp/04-evaluation/data/ground_truth-new.csv" groud_truth-new.csv

In [57]:
!pwd

/home/hsu/Documents/Learning/llm-zoomcamp-hw/module_4


,question,document
0,Is it okay to join the course late if I just f...,74eb249bbf
1,Can I still take this course even if I missed ...,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,I’m a bit late to the course—what do I need to...,74eb249bbf


## Search Evaluation

In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv('data/ground_truth-new.csv')
df_ground_truth.head()

,question,document
0,Is it okay to join the course late if I just f...,74eb249bbf
1,Can I still take this course even if I missed ...,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,I’m a bit late to the course—what do I need to...,74eb249bbf


In [61]:
ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth

[{'question': 'Is it okay to join the course late if I just found it now?',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'I’m a bit late to the course—what do I need to do to still earn the certificate?',
  'document': '74eb249bbf'},
 {'question': 'I registered for the LLM Zoomcamp — when should I expect a confirmation email?',
  'document': '977bf7786c'},
 {'question': 'Do I actually need an acceptance email before I can start the course and hand in homework?',
  'document': '977bf7786c'},
 {'question': 'If I filled out the registration form, does that mean I’m officially on a checked list for the course?',
  'document': '977b

In [62]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [63]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [64]:
q = ground_truth[0]
q

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [65]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [66]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
acf8fa5356 == 74eb249bbf: False


In [67]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [68]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [69]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Is it okay to join the course late if I just found it now?


[1, 0, 0, 0, 0]

In [70]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where can I watch the live sessions for the course, and how do I ask questions during them?


[1, 0, 0, 0, 0]

In [71]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Why does WSL2 say the model needs more memory even though my PC has enough RAM?


[1, 0, 0, 0, 0]

In [72]:
from tqdm.notebook import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [73]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [74]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [75]:
# generalize the search for whatever search function is provided
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance


In [76]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [77]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [78]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

## Search Evaluation Metrics

In [79]:
cnt = 0

for line in relevance_total:
    if 1 in line:
        cnt = cnt + 1

cnt

257

In [80]:
cnt / len(relevance_total)
# 0.933

0.6506329113924051

In [81]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [83]:
hit_rate(relevance_total)

# This means that search found the correct document for 65.1% of the queries in this example.

0.6506329113924051

### Mean Reciprocal Rank (MRR)

In [84]:
total_score = 0.0

for line in relevance_total:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

226.14999999999998

use rank + 1 because Python counts positions from zero. The first position should score 1/1, and without the + 1 we'd divide by zero.



In [85]:
total_score / len(relevance_total) 

0.5725316455696202


MRR is the average of these scores across all queries. It rewards systems that put the correct document near the top.

In [86]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [87]:
mrr(relevance_total)

0.5725316455696202

In [88]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [89]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6506329113924051, 'mrr': 0.5725316455696202}

# Search Parameter Tuning

In [90]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [91]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.6860759493670886, 'mrr': 0.5862447257383967}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.6911392405063291, 'mrr': 0.5935864978902954}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.6506329113924051, 'mrr': 0.5725316455696202}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.6379746835443038, 'mrr': 0.5515611814345992}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.6253164556962025, 'mrr': 0.5332067510548525}


In [92]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [93]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/395 [00:00<?, ?it/s]

In [94]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
34,5.0,10.0,0.2,0.754430,0.646540
18,2.0,4.0,0.1,0.754430,0.646203
33,5.0,10.0,0.1,0.751899,0.646034
3,1.0,2.0,0.1,0.749367,0.645105
35,5.0,10.0,0.5,0.749367,0.645105
19,2.0,4.0,0.2,0.749367,0.645105
4,1.0,2.0,0.2,0.746835,0.643080
20,2.0,4.0,0.5,0.746835,0.642236
7,1.0,4.0,0.2,0.746835,0.633460
6,1.0,4.0,0.1,0.746835,0.633249


In [95]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )